<a href="https://colab.research.google.com/github/PunamBhatia/LogicMojo-AI-ML-April26-Punam/blob/main/RAG/RAGHarborDeskExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!python --version



In [ ]:
# Install LangChain core and community utilities
!pip install -q langchain langchain-community

# Install the explicit Chroma vector database integration
!pip install -q langchain-chroma

# Install the explicit Ollama integration for Chat and Embeddings
!pip install -q langchain-ollama

In [ ]:

from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = Path("/content/drive/MyDrive/LogicMojo/RAGExampleHarborDesk")
CHAT_MODEL = "qwen3.5:4b"               # writes answers
EMBED_MODEL = "qwen3-embedding:4b"      # turns text into numbers

# temperature 0: pick the most likely next word, so the answer does not wander.
llm = ChatOllama(model=CHAT_MODEL, temperature=0, seed=7, num_ctx=4096, reasoning=False, num_predict=180)

print("policy files:", len(list(CORPUS.glob("*.md"))))



In [ ]:
# 1. Install zstd (required by Ollama installer) and then install Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Give the server a few seconds to initialize

# 3. Pull the required model explicitly
!ollama pull qwen3.5:4b

# The question we will reuse in every later step.
question = "How many annual leave days does a full-time HarborDesk employee receive?"

# .invoke sends the string. .content is the reply text.
reply = llm.invoke(question + "\nIf you do not have HarborDesk's documents, say so.")
print(reply.content)

In [ ]:
# Verify if the models have been successfully downloaded/pulled
!ollama list

In [ ]:
#lets load documents
def load_policy(path: Path) -> Document:
  lines = path.read_text(encoding="utf-8").splitlines()
  header, body = lines[:4], lines[4:]
  meta = {"source": path.name}
  #lets read header first and add it in meta
  for line in header:
    key, value = line.split(":", 1)
    meta[key.strip().lower().replace(" ", "_")] = value.strip()
  return Document(page_content="\n".join(body).strip(), metadata=meta)

In [ ]:
#function to chunk the document
def make_chunks(policies:list[Document], chunk_size:int=500)->list[Document]:
  #lets chunk each document and add meta data to each chunk for identification
  splitter= RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=100,
      separators=["\n##","\n\n","\n"," "])

  chunks= splitter.split_documents(policies)
  for chunk in chunks:
    m= chunk.metadata
    chunk.page_content=(f"Document {m['doc_id']} ({m['status']},{m['year']}). {m['title']}.\n"
    +chunk.page_content)

  return chunks



In [ ]:
# Redefining load_policy to fix the TypeError typo
def load_policy(path: Path) -> Document:
  lines = path.read_text(encoding="utf-8").splitlines()
  header, body = lines[:4], lines[4:]
  # FIXED: Use a colon ':' instead of a comma ',' to initialize this as a dictionary
  meta = {"source": path.name}
  # Read header lines and add them to metadata
  for line in header:
    key, value = line.split(":", 1)
    meta[key.strip().lower().replace(" ", "_")] = value.strip()
  return Document(page_content="\n".join(body).strip(), metadata=meta)

policies = [load_policy(path) for path in CORPUS.glob("*.md")]
chunks = make_chunks(policies)

print(f"{len(policies)} policies, {len(chunks)} chunks\n")

for policy in policies:
    print(f"{policy.metadata['doc_id']:16} {policy.metadata['status']:9} {policy.metadata['title']}")

In [ ]:
TASK = "Given a HarborDesk employee question, retrieve the policy passage that answers it"

class QwenEmbeddings(Embeddings):

  def __init__(self):
    self.model = OllamaEmbeddings(model=EMBED_MODEL)

  def embed_query(self, text: str) -> list[float]:
    return self.model.embed_query(f"Instruct:{TASK}\nQuery: {text}")

  def embed_documents(self, texts: list[str]) -> list[list[float]]:
    return self.model.embed_documents(texts)


In [ ]:
# Pull the embedding model before embedding the query
!ollama pull qwen3-embedding:4b

embeddings= QwenEmbeddings()
query_vect= embeddings.embed_query(question)
print(len(query_vect))

In [ ]:
#lets call Chroma.from_documents calls embed_documents on every chunk and saves the index in chroma_db
store = Chroma.from_documents(
    documents=chunks,
    embedding= embeddings,
    ids=[f"student-{i}" for i in range(len(chunks))],  # fixed ids: re-running replaces rows
    collection_name="harbordesk_student",               # separate from the class notebook's collection
    persist_directory="chroma_db",
    collection_metadata={"hnsw:space": "cosine"},       # rank by cosine similarity)
)

#lets search in vectorDB for first 4 chunks with highest similarity search
def search(question:str, k:int=4)->list[Document]:
  hits= store.similarity_search_with_relevance_scores(question, k=k)
  print("Quest:", question)

  for rank, (doc,score) in enumerate(hits,start=1):
      preview= doc.page_content.splitlines()[1][:70]
      print(f"  {rank}. {doc.metadata['doc_id']:16} {score:.3f}  {preview}")

  return [doc for doc,_ in hits]


leave_hits=search(question)




In [ ]:
#lets fetch answer from the chunks
#pass the retrieved chunks into a prompt, then call the same ChatOllama
#STRICT is the contract
STRICT = """You are a HarborDesk policy assistant.
Use only the passages. Cite the document id for every fact, like [HR-LEAVE-2026].
If a passage is archived, do not state it as the current rule.
If the passages do not contain the answer, reply exactly: NOT_IN_CORPUS"""

def answer(question:str, hits:list[Document])->str:
    passages = "\n\n---\n\n".join(doc.page_content for doc in hits)
    prompt = f"{STRICT}\n\nPassages:\n{passages}\n\nQuestion: {question}"
    return llm.invoke(prompt).content

print(answer(question, leave_hits))

In [ ]:
#using langchain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# as_retriever wraps the Chroma store.k=4 is top 4
retriever = store.as_retriever(search_kwargs={"k": 4})

#Join the way answer() did, with --- between them.
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(STRICT + "\n\nPassages:\n{context}\n\nQuestion: {question}")

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke(question))
print()
print(chain.invoke("How long is the HarborDesk sabbatical?"))